<div class="alert alert-block alert-info">

### Imports & Connection
</div>

In [1]:
import os
import numpy as np
import pandas as pd
import seaborn as sns
import plotly.express as px
import matplotlib.pyplot as plt
import datetime as datetime
import plotly.graph_objects as go

from scipy import stats
from pyhive import presto
from IPython.display import display
from datetime import datetime, timedelta
# from scipy.stats import gaussian_kde

import warnings
warnings.filterwarnings('ignore')

In [2]:
pd.set_option('display.max_columns', 50)
pd.set_option('display.max_rows', 100)

In [3]:
## Connection
connection = presto.connect(
        host='presto-gateway.serving.data.production.internal',
        port=80,
        protocol='http',
        catalog='hive',
        username='manoj.ravirajan@rapido.bike'
)

In [4]:
print('Notebook dir - ', os.getcwd())
local_extract = '/Users/E2074/local-datasets/customer/ads-monetisation/monet-benchmark'
print('Data dir - ', local_extract)

Notebook dir -  /Users/E2074/analytics/customer/Ads-Monetisation/monet-benchmark
Data dir -  /Users/E2074/local-datasets/customer/ads-monetisation/monet-benchmark


<div class="alert alert-block alert-info">
    
## Datasets & Parameter
</div>

In [13]:
## Parameter 
start_date = '20241104'
end_date = '20241104'

In [6]:
def func_ad_response(date,segment_date):
    
    
    base_query = f"""


    with response AS (
        
            SELECT
                res.yyyymmdd as yyyymmdd,
                res.city as city,
                res.os as os,
                res.responsetype as responsetype,  
                res.userid as userid,
                res.slotname as slotname, 
                res.format as format, 
                res.responsestatus as responsestatus,
                res.isprefetched as isprefetched,
                coalesce(res.aduuid, res.adid) as response_id,
                REPLACE(REGEXP_EXTRACT(metadata1, 'title=([^,]*)'), 'title=', '') AS title,
                REPLACE(REGEXP_EXTRACT(metadata1, 'advertiserName=([^,]*)'), 'advertiserName=', '') AS advertiserName,
                REPLACE(REGEXP_EXTRACT(metadata1, 'body=([^,]*)'), 'body=', '') AS body
            FROM 
                canonical.iceberg_log_telemetry_ads_impressions_immutable res
            WHERE 
                res.yyyymmdd = '{date}'
                AND res.eventName = 'Ad_Response'
                AND NOT regexp_like(res.format, 'Arrived|OTW|Started|nativeBanner')
                AND res.responsetype IN ('GAMBanner', 'ONLINE_SALES', 'System Banner')
                AND (res.responsestatus = 'true' OR res.responsestatus IS NULL)

                -- AND quarter_hour = '1115'
        ),

        segment as (

        SELECT
            customer_id,
            taxi_ltr_segment,
            taxi_retention_segments taxi_retention_segment,
            taxi_regularity_segment taxi_rr_regularity_segment,
            taxi_intent_segment taxi_rr_intent_segment,
            taxi_fe_regularity_segment,
            taxi_fe_intent_segment
        FROM 
            datasets.iallocator_customer_segments
        WHERE 
            run_date = '{segment_date}'
        )
        
        SELECT
            response.yyyymmdd as yyyymmdd,
            response.city as city,
            response.os as os,
            response.responsetype as responsetype,  
            response.userid as userid,
            response.slotname as slotname, 
            response.format as format, 
            response.responsestatus as responsestatus,
            response.isprefetched as isprefetched,
            
            taxi_ltr_segment,
            taxi_retention_segment,
            taxi_rr_regularity_segment,
            taxi_rr_intent_segment,
            taxi_fe_regularity_segment,
            taxi_fe_intent_segment,

            response_id,
            
            title,
            advertiserName,
            body
            
        FROM    
            response
        LEFT JOIN 
            segment as segment
            ON response.userid = segment.customer_id

    """
    
    df_base = pd.read_sql(base_query, connection)

    return df_base



In [7]:
def func_ad_impression(date):
    
    
    base_query = f"""
    
        SELECT
            DISTINCT
            coalesce(imp.aduuid, imp.adid) as impression_id,
            imp.userid as userid,
            imp.yyyymmdd as yyyymmdd
        FROM 
            canonical.iceberg_log_telemetry_ads_impressions_immutable as imp
        WHERE 
            imp.yyyymmdd = '{date}'
            AND imp.eventName = 'Ad_Impression'
            
            -- AND quarter_hour = '1115'

    """
    
    df_base = pd.read_sql(base_query, connection)

    return df_base

In [8]:
def func_ad_viewable_impression(date):
    
    
    base_query = f"""
    
        SELECT
            DISTINCT
            coalesce(view_imp.aduuid, view_imp.adid) as viewable_impression_id,
            view_imp.userid as userid,
            view_imp.yyyymmdd as yyyymmdd
        FROM 
            canonical.iceberg_log_telemetry_ads_impressions_immutable view_imp
        WHERE 
            view_imp.yyyymmdd = '{date}'
            AND view_imp.eventName = 'Ad_Viewable_Impression'
            
            -- AND quarter_hour = '1115'

    """
    
    df_base = pd.read_sql(base_query, connection)

    return df_base

In [9]:
def func_ad_click(date):
    
    
    base_query = f"""

            
        SELECT
            DISTINCT
            coalesce(clk.aduuid, clk.adid) as click_id,
            clk.userid as userid,
            clk.yyyymmdd as yyyymmdd
        FROM 
            canonical.iceberg_log_telemetry_ads_impressions_immutable as clk
        WHERE 
            clk.yyyymmdd = '{date}'
            AND clk.eventName = 'Ad_Click'
            
            -- AND quarter_hour = '1115'

    """
    
    df_base = pd.read_sql(base_query, connection)

    return df_base

In [10]:
def func_pull_ads_data(start_date, end_date):
    
    date_list = pd.date_range(start=start_date, end=end_date).strftime('%Y%m%d').tolist()
    formatted_end_date = datetime.strptime(end_date, '%Y%m%d').strftime('%Y-%m-%d')
    
    for date in date_list:
        
        df_response_data_dump = func_ad_response(date, formatted_end_date)
        # display(df_response_data_dump.head(5))
        
        df_impression_data_dump = func_ad_impression(date)
        # display(df_impression_data_dump.head(5))
        
        df_viewable_impression_data_dump = func_ad_viewable_impression(date)
        # display(df_viewable_impression_data_dump.head(5))
        
        df_click_data_dump = func_ad_click(date)
        # display(df_click_data_dump.head(5))
        
        merge1 = pd.merge(df_response_data_dump, df_impression_data_dump, 
                          how='left', 
                          left_on=['yyyymmdd', 'userid', 'response_id'],
                          right_on=['yyyymmdd', 'userid', 'impression_id'])
        merge2 = pd.merge(merge1, df_viewable_impression_data_dump, 
                          how='left', 
                          left_on=['yyyymmdd', 'userid', 'impression_id'],
                          right_on=['yyyymmdd', 'userid', 'viewable_impression_id'])
        merge3 = pd.merge(merge2, df_click_data_dump, 
                          how='left', 
                          left_on=['yyyymmdd', 'userid', 'response_id'],
                          right_on=['yyyymmdd', 'userid', 'click_id'])
        merge3.to_parquet(local_extract + '/data_dump/{}_data.parquet'.format(date), index=False)

    folder_path = local_extract + '/data_dump'
    parquet_files = [os.path.join(folder_path, file) for file in os.listdir(folder_path) if file.endswith('.parquet')]
    df = pd.concat([pd.read_parquet(file) for file in parquet_files], ignore_index=True)
    
    return df

In [11]:
def func_percentage(df, numerator, denominator):
    df['ctr'] = (df[numerator]*100.00/df[denominator]).round(2)
    return df

In [ ]:
df = func_pull_ads_data(start_date, end_date)

In [ ]:
df

In [24]:
df_merge

,yyyymmdd,city,os,responsetype,userid,slotname,format,responsestatus,isprefetched,taxi_ltr_segment,taxi_retention_segment,taxi_rr_regularity_segment,taxi_rr_intent_segment,taxi_fe_regularity_segment,taxi_fe_intent_segment,response_id,title,advertiserName,body,impression_id,viewable_impression_id,click_id
0,20241104,Hyderabad,android,GAMBanner,5bf7a526647b2949efed5dde,PostOrderStarted1,gamCombinedBanner,None,None,PHH,PLATINUM,WEEKLY,MEDIUM,BI_WEEKLY,LOW,ad_5bf7a526647b2949efed5dde_1730688557024,None,None,None,ad_5bf7a526647b2949efed5dde_1730688557024,ad_5bf7a526647b2949efed5dde_1730688557024,NaN
1,20241104,Chennai,android,GAMBanner,635ba81785b399c6745338e7,PostOrderStarted3,gamCombinedBanner,None,None,PHH,GOLD,BI_WEEKLY,LOW,BI_WEEKLY,MEDIUM,ad_635ba81785b399c6745338e7_1730699397815,None,None,None,NaN,NaN,NaN
2,20241104,Madurai,android,System Banner,629cb57a1b57e48b7be81a8e,PostOrderStarted4,Rewards,None,None,PHH,GOLD,BI_WEEKLY,LOW,MONTHLY,LOW,ad_629cb57a1b57e48b7be81a8e_1730698942761,None,None,None,ad_629cb57a1b57e48b7be81a8e_1730698942761,ad_629cb57a1b57e48b7be81a8e_1730698942761,NaN
3,20241104,Hyderabad,android,GAMBanner,638b4409d1162513e22529db,PostOrderStarted3,gamCombinedBanner,true,false,PHH,PLATINUM,BI_WEEKLY,MEDIUM,BI_WEEKLY,LOW,b69c5814-6748-490d-9c1f-9c1a8ed358eb638b4409d1...,None,None,None,NaN,NaN,NaN
4,20241104,Hyderabad,android,GAMBanner,638b4409d1162513e22529db,PostOrderStarted2,gamCombinedBanner,true,false,PHH,PLATINUM,BI_WEEKLY,MEDIUM,BI_WEEKLY,LOW,10b46b16-bf50-4902-8a86-3fb4783fb244638b4409d1...,None,None,None,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
140577,20241104,Jaipur,android,GAMBanner,6682bd3191ae397ce7b57d94,CaptainSearch1,gamCombinedBanner,true,false,UNKNOWN,UNKNOWN,UNKNOWN,UNKNOWN,UNKNOWN,UNKNOWN,b91db866-06fe-46d9-83d0-8e350fb85caf6682bd3191...,None,None,None,NaN,NaN,NaN
140578,20241104,Warangal,android,GAMBanner,6507166192c3a879f59c432b,PostOrderStarted3,gamCombinedBanner,true,false,PHH,ELITE,DAILY,HIGH,DAILY,HIGH,6fdf435a-60e9-4ffb-a2cb-46d06aaa6c306507166192...,None,None,None,NaN,NaN,NaN
140579,20241104,Warangal,android,GAMBanner,6507166192c3a879f59c432b,PostOrderStarted1,gamCombinedBanner,true,false,PHH,ELITE,DAILY,HIGH,DAILY,HIGH,54546ff2-44da-4afe-a085-70c4599b1b8f6507166192...,None,None,None,NaN,NaN,NaN
140580,20241104,Hyderabad,android,GAMBanner,6402279d8e212cf5464d3338,PostOrderStarted3,gamCombinedBanner,true,false,PHH,SILVER,MONTHLY,LOW,UNKNOWN,UNKNOWN,4db694ef-3f15-400c-bee3-4d16ec224b286402279d8e...,None,None,None,NaN,NaN,NaN


In [43]:
df_overall = df_merge\
.groupby(['yyyymmdd','os', 'responsetype'])\
.agg(
    response = ('response_id', 'nunique'),
    impression = ('impression_id', 'nunique'),
    viewable_impression = ('viewable_impression_id', 'nunique'),
    click = ('click_id', 'nunique')
)\
.reset_index()

func_percentage(df_overall, 'click', 'viewable_impression')
df_overall

,yyyymmdd,os,responsetype,response,impression,viewable_impression,click,ctr
0,20241104,android,GAMBanner,79054,13806,11292,76,0.67
1,20241104,android,ONLINE_SALES,4493,4036,1806,12,0.66
2,20241104,android,System Banner,44897,38992,20485,66,0.32
3,20241104,iOS,ONLINE_SALES,888,602,130,9,6.92
